## <center style="color:blue;">**FootVerse**</center>

### <center>**Modélisation et Analyse de Données Footballistiques**</center>

Ce projet a pour objectif de collecter des données de football à l’aide du web scraping (Selenium), puis de les transformer et nettoyer afin d’assurer leur qualité. Les données seront ensuite modélisées et stockées dans une base de données PostgreSQL. Enfin, un modèle de machine learning sera entraîné pour prédire l’équipe gagnante d’un match.

### <span style="color:green;">**Environnement Selinium :**</span>

#### <span style="color:orange;">**1. Selinium :**</span>

##### **1.1. Définition :**

``Selenium`` est un outil d’automatisation des navigateurs web.

Il permet à un programme (en Python, Java, JavaScript, etc.) de contrôler un navigateur comme le ferait un humain :
- Ouvrir un site.
- Cliquer sur des boutons
- Remplir des formulaires
- Faire des recherches
- Récupérer des données 
- etc..

##### **1.2. A quoi Sert ?**

Selenium est utilisé dans plusieurs cas :

1. Tester automatiquement des sites web
    - ***Exemple :*** vérifier que ton formulaire de connexion fonctionne sans erreur après chaque mise à jour.

1. Automatiser des actions répétitives
    - ***Exemple :*** remplir un même formulaire des centaines de fois, télécharger des fichiers automatiquement, etc.

1. Faire du web scraping dynamique
    - ***Exemple :*** extraire des données d’un site qui charge son contenu avec JavaScript (ce que requests ne peut pas faire facilement).

1. Former des bots web (à but légal bien sûr)
    - ***Exemple :*** un bot qui simule un utilisateur sur un site pour tester les performances.

##### **1.3. Fonctionnement :**

Le fonctionnement de ``Selenium`` repose sur trois éléments principaux :

```bash
Code Python  ->  Selenium  ->  WebDriver  ->  Navigateur
```

Etapes :

1. Le script Python (ou Java, etc.) utilise la bibliothèque ``Selenium``.

2. ``Selenium`` envoie des commandes au navigateur via un intermédiaire appelé ``WebDriver``.

3. ``WebDriver`` traduit ces commandes en actions réelles sur le navigateur (ouvrir une page, cliquer, etc.).

4. Le navigateur exécute ces actions et renvoie les résultats à ``Selenium``.

##### **1.4. Installation :**

Pour installer ``Selenium`` pour Python, on utilise la commande suivante :

```bash
pip install selenium
```

Et pour vérifier que Selenium est installé avec succés, on utilise la commande :

```bash
pip show selenium
```

#### <span style="color:orange;">**2. WebDriver :**</span>

##### **2.1. Définition :**

Le ``WebDriver`` est le pont de communication entre Selenium et le navigateur web.

Chaque navigateur a son propre ``WebDriver`` :

| Navigateur | WebDriver    |
| ---------- | ------------ |
| Chrome     | ChromeDriver |
| Firefox    | GeckoDriver  |
| Edge       | EdgeDriver   |
| Safari     | SafariDriver |


##### **2.2. A quoi Sert ?**

Il sert à traduire les commandes Selenium en actions réelles dans le navigateur.

Exemple :

```bash
driver.get("https://www.google.com")
```

- Selenium envoie cette commande à ChromeDriver
- ChromeDriver demande à Chrome d’ouvrir l’URL
- Chrome ouvre la page et renvoie la réponse à ton programme.

##### **2.3. Fonctionnement :**

- Selenium lance un serveur WebDriver en arrière-plan.

- Ce serveur écoute les commandes HTTP (comme “ouvre cette page”, “clique ici”, etc.).

- Le navigateur exécute ces commandes via une API standardisée appelée WebDriver Protocol.

- Les résultats (comme les éléments trouvés ou les messages d’erreur) sont renvoyés à Selenium.

<br>

### <span style="color:green;">**Scrapping :**</span>

#### <span style="color:orange;">**1. Définition du WebDriver et le Site Web à Scrapper :**</span>

In [124]:
from selenium import webdriver

website = "https://fbref.com/en/comps/9/history/Premier-League-Seasons"

driver = webdriver.Chrome()

driver.get(website)

#### <span style="color:orange;">**2. Extraction du Saison - 2024/2025 :**</span>

In [125]:
from selenium.webdriver.common.by import By
import pandas as pd

season_head = driver.find_elements(By.CSS_SELECTOR, "table#seasons thead tr th")

season_infos = driver.find_elements(By.CSS_SELECTOR, "table#seasons tbody tr[data-row='1']")

season = season_infos[0].find_elements(By.CSS_SELECTOR, "th a")
saison = season[0].text

stats = season_infos[0].find_elements(By.CSS_SELECTOR, "td")

headers = [head.text for head in season_head]

values = [saison] + [stat.text for stat in stats]

df = pd.DataFrame([values], columns=headers)

df.to_csv("../data/bronze/saisons.csv", index=False)


#### <span style="color:orange;">**3. Extraction des Equipes :**</span>

In [126]:
season[0].click()

In [127]:
teams_data = []

teams_table = driver.find_element(By.CSS_SELECTOR, "table#results2024-202591_overall")

teams_heads = teams_table.find_elements(By.CSS_SELECTOR, "thead tr th")[:18]

team_head = list(map(lambda x: x.text, teams_heads))

teams_infos = teams_table.find_elements(By.CSS_SELECTOR, "tbody tr")

for info in teams_infos :

    data = info.find_elements(By.CSS_SELECTOR, "th,td")[:18]

    team_data = list(map(lambda x: x.text, data))

    teams_data.append(team_data)

df = pd.DataFrame(data=teams_data, columns=team_head)

df.to_csv(f"../data/bronze/teams.csv", index=False)

#### <span style="color:orange;">**4. Extraction des Joueurs et Statistiques des Matchs de chaque Equipe :**</span>

In [ ]:
import time
import os

# Switcher vers la 1ère fenêtre
driver.switch_to.window(driver.window_handles[0])

# Les Noms et les Liens des Equipes
teams = driver.find_elements(By.CSS_SELECTOR, "table#results2024-202591_overall tbody tr td[data-stat='team'] a")

for team in teams:

    team_name = team.text

    # Création des Dossiers de Chaque Equipe 
    os.makedirs(f"../data/bronze/teams/{team_name}", exist_ok=True)

    # List pour stocker les joueurs de chaque équipe
    players_data = []

    # List pour stocker les statistiques des matchs de chaque équipe
    scores_data = []

    # Ouvrir une nouvelle Fenêtre avec url de l'équipe séléctionné
    driver.execute_script(f"window.open('{team.get_attribute("href")}')")
    # Switcher vers la nouvelle Fenêtre
    driver.switch_to.window(driver.window_handles[-1])

    time.sleep(3)

    # Extraction des Joueurs de Chaque Equipe et les Enregistrer sous Format csv
    squad_table = driver.find_element(By.CSS_SELECTOR, "table#stats_standard_9")    
  
    squad_heads = squad_table.find_elements(By.CSS_SELECTOR, "thead tr:last-child th")[:16]

    squad_head = list(map(lambda x: x.text, squad_heads))

    squad_infos = squad_table.find_elements(By.CSS_SELECTOR, "tbody tr:not([class])")

    for info in squad_infos:

        data = info.find_elements(By.CSS_SELECTOR, "th,td")[:16]

        player_data = list(map(lambda x: x.text, data))

        players_data.append(player_data)

    df = pd.DataFrame(data=players_data, columns=squad_head)

    df.to_csv(f"../data/bronze/teams/{team_name}/players.csv", index=False)    

    
    
    

    # Extraction des Statistiques des Matchs de Chaque Equipe et les Enregistrer sous Format csv
    scores_table = driver.find_element(By.CSS_SELECTOR, "table#matchlogs_for")    
  
    scores_heads = scores_table.find_elements(By.CSS_SELECTOR, "thead tr th")[:18]

    score_head = list(map(lambda x: x.text, scores_heads))

    scores_infos = scores_table.find_elements(By.CSS_SELECTOR, "tbody tr:not([class])")

    for info in scores_infos:

        data = info.find_elements(By.CSS_SELECTOR, "th,td")[:18]

        score_data = list(map(lambda x: x.text, data))

        scores_data.append(score_data)
    
    df = pd.DataFrame(data=scores_data, columns=score_head)

    df.to_csv(f"../data/bronze/teams/{team_name}/scores.csv", index=False)

    # Fermer la Fenêtre Actuelle
    driver.close()

    # Revenir à la 1ère Fenêtre
    driver.switch_to.window(driver.window_handles[0])